# Chapter 5 — A Minimal Code-Executing Agent

Hands-on lab: build the backbone agent, solve three small tasks (a math
problem, a file transform, an API-free data task) — and then go past a
single run into three deeper, real-live-model measurements: multi-trial
reliability, a prompt ablation, and a real step-budget boundary.

Every response below is real output from `groq/llama-3.3-70b-versatile` via
litellm — no scripted text anywhere in this notebook. Requires
`GROQ_API_KEY` in the environment.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent.parent / "src"))
sys.path.insert(0, str(Path.cwd().parent / "code"))

from backbone_agent import run_agent
from backbone_agent.loop import SYSTEM_PROMPT

print(SYSTEM_PROMPT)

You are a coding agent. Solve the task by writing and running Python code.

On each turn:
- If you need to compute, inspect, or produce something, respond with a single ```python code block. It will be executed, and you will see its stdout (or an error traceback) as the next message.
- When the task is fully solved, respond with plain text and NO code block, stating the final answer. This ends the task.

Rules:
- Exactly one code block per turn — put all the logic for this step in it.
- Use print(...) for anything you need to see back; only printed output becomes visible to you.
- If your code raises an error, read the traceback in the next message, fix the code, and try again.



## 1. Smallest possible run

One trivial task, full trace shown, before the deeper measurements.

In [2]:
answer, messages = run_agent("What is 17 * 23? Compute it, don't just guess.", return_trace=True)
for m in messages:
    if m["role"] == "system":
        continue
    print(f"--- {m['role']} ---")
    print(m["content"].strip())
    print()
print("FINAL ANSWER:", answer)

--- user ---
What is 17 * 23? Compute it, don't just guess.

--- assistant ---
```python
result = 17 * 23
print(result)
```

--- user ---
Observation:
391

--- assistant ---
The result of 17 * 23 is indeed 391.

FINAL ANSWER: The result of 17 * 23 is indeed 391.


## 2. The three required hands-on tasks (one run each)

In [3]:
import time
from three_tasks_demo import task_1_math, task_2_file_transform, task_3_data_stats, render_trace

r1 = task_1_math()
time.sleep(2)  # pace requests under Groq's free-tier TPM limit
r2 = task_2_file_transform()
time.sleep(2)
r3 = task_3_data_stats()
for r in (r1, r2, r3):
    print(f"{r['name']}: expected={r['expected']}  success={r['success']}")

math (sum of first 20 primes): expected=639  success=True
file transform (average a CSV column): expected=79.6  success=True
data task (mean/median/stdev of an inline list): expected=(21, 18, 12.96)  success=True


## 3. Multi-trial reliability — is one run enough to trust?

A single success doesn't tell you whether a task is reliably solvable or
just got lucky once. Run the math and data-stats tasks 3 times each,
live, and report a real success rate — our own small pass@3-style
measurement.

In [4]:
from reliability_and_ablation import run_reliability_trials

math_reliability = run_reliability_trials(task_1_math, n_trials=3)
print(f"math task:  {math_reliability['n_success']}/{math_reliability['n_trials']} "
      f"({math_reliability['success_rate']:.0%}), step counts: {math_reliability['step_counts']}")

stats_reliability = run_reliability_trials(task_3_data_stats, n_trials=3)
print(f"stats task: {stats_reliability['n_success']}/{stats_reliability['n_trials']} "
      f"({stats_reliability['success_rate']:.0%}), step counts: {stats_reliability['step_counts']}")

math task:  3/3 (100%), step counts: [2, 2, 2]


stats task: 3/3 (100%), step counts: [3, 3, 3]


Both tasks solved correctly on every trial, with identical step counts
each time — these are simple enough tasks that the model's behavior is
highly consistent. This won't be true for every task (harder or more
ambiguous tasks would show real variance across trials); the value of
running multiple trials is finding OUT whether variance exists, not
assuming there is or isn't any.

## 4. Prompt ablation — does steering the model away from pandas/numpy help?

Chapter 5's original run found the model's first code action for the
file-transform task commonly reaches for `pandas`, which isn't installed,
producing a real `ModuleNotFoundError` it then recovers from. Does a single
added sentence in the system prompt — telling the model to prefer the
standard library — actually reduce how often that happens? A/B it directly:
3 live trials with the original prompt, 3 with a one-sentence variant.

In [5]:
from reliability_and_ablation import ALT_SYSTEM_PROMPT, run_prompt_ablation

print("Added sentence:")
print(ALT_SYSTEM_PROMPT[len(SYSTEM_PROMPT):])

Added sentence:

Prefer Python's standard library (e.g. csv, statistics, json) over third-party packages like pandas or numpy — assume third-party packages are NOT installed unless you have already confirmed otherwise this run.


In [6]:
time.sleep(10)  # let the TPM window recover between experiment sections
ablation = run_prompt_ablation(n_trials=3)
for label, stats in ablation.items():
    print(f"{label}:")
    print(f"  success rate:              {stats['n_success']}/{stats['n_trials']} ({stats['success_rate']:.0%})")
    print(f"  ModuleNotFoundError rate:  {stats['module_not_found_rate']:.0%}")
    print(f"  avg steps:                 {stats['avg_steps']:.1f}  (step counts: {stats['step_counts']})")
    print()

default_prompt:
  success rate:              3/3 (100%)
  ModuleNotFoundError rate:  100%
  avg steps:                 3.0  (step counts: [3, 3, 3])

stdlib_steered_prompt:
  success rate:              3/3 (100%)
  ModuleNotFoundError rate:  0%
  avg steps:                 2.0  (step counts: [2, 2, 2])



**A one-sentence prompt change eliminated the failure mode entirely** in
this run: the default prompt hit `ModuleNotFoundError` on 3/3 trials
(needing 3 steps every time — the wasted `pandas` attempt, then the real
fix); the stdlib-steered prompt hit it on 0/3 trials, completing in the
minimum possible 2 steps every time. Both prompts reached the CORRECT final
answer either way (100% success rate for both) — the prompt change didn't
fix a correctness problem, it fixed an *efficiency* problem caused by the
model's own default library preference colliding with this specific
minimal environment. This is exactly the kind of result Chapter 44
("Prompting for Reliable Code") will generalize — a concrete, measured
instance of it, not a preview promise.

## 5. A real step-budget boundary

`run_agent` raises `StepBudgetExceeded` if no final answer is reached
within `max_steps`. Even a "clean" run of the file-transform task needs at
least 2 steps (one code action, one separate final-answer turn) — so
`max_steps=1` should make even a perfectly-behaved run fail. Confirm this
for real, not just in theory.

In [7]:
from backbone_agent.loop import StepBudgetExceeded

task = "Compute the sum of the first 20 prime numbers. State the final numeric answer clearly."
try:
    run_agent(task, max_steps=1)
    print("did not raise (unexpected)")
except StepBudgetExceeded as e:
    print(f"StepBudgetExceeded raised as expected: {e}")

StepBudgetExceeded raised as expected: no final answer within 1 steps
